# MAPPO Random Wall Proximity Source Curriculum 50x50 Outer / 30x30 Inner


In [1]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "ant_byte_env").exists():
    raise RuntimeError("Launch this notebook from the cool-antz repo or a subdirectory.")
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
runtime_status


/home/narf/miniconda3/envs/cool-antz/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


{'jax_already_imported': False,
 'jax_preallocate': 'false',
 'jax_memory_fraction': '0.35',
 'jax_allocator': 'platform',
 'memory_trimmed': True,
 'disk_free_gb': 19.26,
 'disk_used_percent': 81.7,
 'current_pid': 43700,
 'safe_cleanup_candidate_count': 10,
 'safe_cleanup_candidate_gb': 0.002,
 'top_memory_processes': [{'pid': 22140,
   'ppid': 3894,
   'rss_mb': 1949.6,
   'command': '/home/narf/miniconda3/envs/cool-antz/bin/python -m ipykernel_launcher --f=/run/user/1000/jupyter/runtime/kernel-v3add8fa893a84492840e625cc877e8a7b8f5479c8.json',
   'connection_file': '/run/user/1000/jupyter/runtime/kernel-v3add8fa893a84492840e625cc877e8a7b8f5479c8.json',
   'is_current_process': False,
   'is_notebook_kernel': True},
  {'pid': 7581,
   'ppid': 2684,
   'rss_mb': 1253.4,
   'command': '/snap/firefox/8585/usr/lib/firefox/firefox',
   'connection_file': None,
   'is_current_process': False,
   'is_notebook_kernel': False},
  {'pid': 6387,
   'ppid': 3894,
   'rss_mb': 1223.2,
   'command

In [2]:
import importlib

import jax

from ant_byte_env.training.jax_mappo import runner as jax_runner

workflows = importlib.reload(workflows)
jax_runner = importlib.reload(jax_runner)
print(f"JAX device: {jax.devices()[0]}")


JAX device: cuda:0


## Quick Smoke Run

Run one tiny job before starting the scratch curriculum.

In [3]:
smoke_metrics = workflows.run_jax_smoke(jax_runner.main)
smoke_metrics


{'loss': -0.006546946242451668,
 'policy_loss': -2.9802322387695312e-08,
 'value_loss': 0.03295765072107315,
 'entropy': 2.3025741577148438,
 'approx_kl': 0.0,
 'clipfrac': 0.0,
 'grad_norm': 0.8305642008781433,
 'episode_return': 0.0,
 'env_return': 0.0,
 'completed_episodes': 1.0,
 'terminated_episodes': 0.0,
 'truncated_episodes': 1.0,
 'pickup_events': 0.0,
 'delivery_events': 0.0,
 'mean_carrying_ants': 0.0,
 'final_mean_remaining_food': 1.0,
 'visited_cell_events': 1.0,
 'mean_visited_cell_count': 4.25,
 'final_mean_visited_cell_count': 5.0,
 'mean_visited_cell_fraction': 0.265625,
 'final_mean_visited_cell_fraction': 0.3125,
 'viewed_cell_events': 0.0,
 'mean_viewed_cell_count': 14.0,
 'final_mean_viewed_cell_count': 14.0,
 'mean_viewed_cell_fraction': 0.875,
 'final_mean_viewed_cell_fraction': 0.875,
 'mean_visible_border_cells': 3.75,
 'final_mean_visible_border_cells': 3.0,
 'mean_border_moat_cost': 0.0,
 'final_mean_border_moat_cost': 0.0,
 'write_action_nonzero_rate': 0.5,


## Curriculum Settings

Edit `experiments/exploration_to_forage_proximity_sources_50x50.json` for durable footprint, reward-scale, margin, or budget changes.

In [4]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "exploration_to_forage_proximity_sources_50x50_random_walls.json"
experiment = workflows.load_jax_experiment(EXPERIMENT_CONFIG)
experiment_args = dict(experiment.args)

RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / experiment.name
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MEDIA_DIR = RUN_DIR / "media"
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
ROLLOUT_POLICY_TEMPERATURE = workflows.notebook_rollout_policy_temperature(experiment.metadata)
WANDB_VIDEO_MAX_FRAMES = int(experiment.metadata["wandb_video_max_frames"])
WANDB_VIDEO_STAGE_NAMES = tuple(experiment.metadata["wandb_preview_stage_names"])
WANDB_VIDEO_ROLLOUT_COUNT = int(experiment.metadata.get("wandb_preview_rollout_count", 1))
CHECKPOINT_VIDEO_INTERVAL_UPDATES = int(experiment.metadata.get("checkpoint_video_interval_updates", 0))
CHECKPOINT_VIDEO_MAX_FRAMES = int(experiment.metadata.get("checkpoint_video_max_frames", WANDB_VIDEO_MAX_FRAMES))
CHECKPOINT_VIDEO_ROLLOUT_COUNT = int(experiment.metadata.get("checkpoint_video_rollout_count", WANDB_VIDEO_ROLLOUT_COUNT))
CHECKPOINT_VIDEO_POLICY_TEMPERATURE = float(experiment.metadata.get("checkpoint_video_policy_temperature", ROLLOUT_POLICY_TEMPERATURE))
CHECKPOINT_VIDEO_WANDB_KEY_PREFIX = experiment.metadata.get(
    "checkpoint_video_wandb_key_prefix",
    "videos/exploration_to_forage/random_walls/checkpoints",
)
STAGE_UPDATE_MULTIPLIER = float(experiment.metadata.get("stage_update_multiplier", 1.0))
SOURCE_CHECKPOINT = None
if experiment_args.get("load_model"):
    SOURCE_CHECKPOINT = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["load_model"])
    if not SOURCE_CHECKPOINT.exists():
        raise FileNotFoundError(f"Run or restore the source checkpoint first: {SOURCE_CHECKPOINT}")
    experiment_args["load_model"] = str(SOURCE_CHECKPOINT)
BEST_CHECKPOINT_PATH = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["save_best_model"])
experiment_args["save_best_model"] = str(BEST_CHECKPOINT_PATH)

SOURCE_COUNTS = tuple(int(count) for count in experiment.metadata["food_source_counts"])
CLUSTER_RADII = tuple(int(radius) for radius in experiment.metadata["food_cluster_radii"])
CURRICULUM_STAGES = workflows.build_food_cluster_curriculum_stages(
    experiment_args,
    source_counts=SOURCE_COUNTS,
    cluster_radii=CLUSTER_RADII,
    visit_reward_schedule=experiment.metadata.get("visit_reward_schedule"),
    view_reward_schedule=experiment.metadata.get("view_reward_schedule"),
    stage_update_multiplier=STAGE_UPDATE_MULTIPLIER,
)
GLOBAL_UPDATE_CAP = max(int(stage["global_update_cap"]) for stage in CURRICULUM_STAGES)
UPDATE_TIMESTEPS = workflows.update_timesteps(
    num_envs=int(experiment_args["num_envs"]),
    num_steps=int(experiment_args["num_steps"]),
)
WANDB_PROJECT = "cool-antz"
WANDB_ENTITY = None
WANDB_GROUP = experiment.name
WANDB_RUN_NAME = WANDB_GROUP
WANDB_MODE = "online"
CRITIC_TAG = f"{experiment_args.get('critic_architecture', 'mlp').replace('_', '-')}-critic"
COMMON_ARGS = workflows.config_common_args(
    experiment_args,
    exclude=workflows.EXPLORATION_TO_FORAGE_ARG_EXCLUDES,
)
{
    "source_checkpoint": SOURCE_CHECKPOINT,
    "best_checkpoint": BEST_CHECKPOINT_PATH,
    "num_ants": experiment_args.get("num_ants"),
    "normal_food_sources": experiment_args.get("food_sources"),
    "lethal_food_sources": experiment_args.get("lethal_food_sources"),
    "lethal_food_distance_range": (
        experiment_args.get("lethal_food_min_distance"),
        experiment_args.get("lethal_food_max_distance"),
    ),
    "random_food_same_distance": experiment_args.get("random_food_same_distance"),
    "random_wall_obstacles": experiment_args.get("random_wall_obstacles"),
    "maze_layout_count": experiment_args.get("maze_layout_count"),
    "random_wall_count_range": (
        experiment_args.get("random_wall_count_min"),
        experiment_args.get("random_wall_count_max"),
    ),
    "random_wall_length_range": (
        experiment_args.get("random_wall_length_min"),
        experiment_args.get("random_wall_length_max"),
    ),
    "random_wall_width": experiment_args.get("random_wall_width"),
    "random_wall_l_turn_probability": experiment_args.get("random_wall_l_turn_probability"),
    "random_wall_center_window_size": experiment_args.get("random_wall_center_window_size"),
    "layout_margin": experiment_args.get("layout_margin"),
    "hub_center_window_size": experiment_args.get("hub_center_window_size"),
    "distance_bonus": experiment_args.get("distance_bonus"),
    "carrying_hub_distance_bonus": experiment_args.get("carrying_hub_distance_bonus"),
    "critic_architecture": experiment_args.get("critic_architecture"),
    "write_bits": experiment_args.get("write_bits"),
    "per_ant_write_channels": experiment_args.get("per_ant_write_channels"),
    "stage_training_profiles": [
        (
            stage["name"],
            stage["food_sources"],
            stage["food_cluster_radius"],
            stage["food_count"],
            stage["global_update_cap"],
            stage["num_steps"],
            stage["gamma"],
        )
        for stage in CURRICULUM_STAGES
    ],
    "total_updates_per_stage": GLOBAL_UPDATE_CAP,
    "update_timesteps": UPDATE_TIMESTEPS,
}


{'source_checkpoint': PosixPath('/home/narf/Desktop/Facultad/RL/cool-antz-nuevo/cool-antz/runs/notebooks/jerf_best_results/best_full_layout_proximity_60ants_half_food_shared_writes_write_cost_8bits_stabilized.pkl'),
 'best_checkpoint': PosixPath('/home/narf/Desktop/Facultad/RL/cool-antz-nuevo/cool-antz/runs/notebooks/fl50_60ants_sparse_far_food_near_lethal_random_walls_from_stabilized_best/checkpoints/best_full_layout_60ants_sparse_far_food_near_lethal_random_walls.pkl'),
 'num_ants': 60,
 'normal_food_sources': 12,
 'lethal_food_sources': 1,
 'lethal_food_distance_range': (2, 7),
 'random_food_same_distance': True,
 'random_wall_obstacles': True,
 'maze_layout_count': 256,
 'random_wall_count_range': (5, 9),
 'random_wall_length_range': (3, 8),
 'random_wall_width': 2,
 'random_wall_l_turn_probability': 0.6,
 'random_wall_center_window_size': 18,
 'layout_margin': 0,
 'hub_center_window_size': 24,
 'distance_bonus': 0.0,
 'carrying_hub_distance_bonus': 0.0,
 'critic_architecture': 'st

## Train Random Wall Curriculum


In [5]:
training_result = workflows.run_forage_curriculum(
    stages=CURRICULUM_STAGES,
    checkpoint_dir=CHECKPOINT_DIR,
    common_args=COMMON_ARGS,
    update_timesteps_per_stage=UPDATE_TIMESTEPS,
    global_update_cap=GLOBAL_UPDATE_CAP,
    train_main=jax_runner.main,
    initial_checkpoint=SOURCE_CHECKPOINT,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
    wandb_group=WANDB_GROUP,
    wandb_run_name=WANDB_RUN_NAME,
    wandb_mode=WANDB_MODE,
    wandb_tags=[
        "exploration-to-forage",
        "random-walls",
        "near-nest-walls",
        "lethal-cookies",
        "single-lethal-cookie",
        "sparse-cookies",
        "2-ants",
        CRITIC_TAG,
        "50x50",
        "30x30-inner",
    ],
    wandb_notes=experiment.metadata["notes"],
    wandb_artifact_paths=[EXPERIMENT_CONFIG],
    wandb_artifact_prefix="exploration-to-forage-random-walls",
    checkpoint_name_prefix="jax_mappo_exploration_to_forage_random_walls",
    wandb_video_key_prefix="videos/exploration_to_forage/random_walls",
    wandb_video_max_frames=WANDB_VIDEO_MAX_FRAMES,
    wandb_video_stage_names=WANDB_VIDEO_STAGE_NAMES,
    wandb_video_policy_temperature=ROLLOUT_POLICY_TEMPERATURE,
    wandb_video_rollout_count=WANDB_VIDEO_ROLLOUT_COUNT,
    checkpoint_video_interval_updates=CHECKPOINT_VIDEO_INTERVAL_UPDATES,
    checkpoint_video_max_frames=CHECKPOINT_VIDEO_MAX_FRAMES,
    checkpoint_video_policy_temperature=CHECKPOINT_VIDEO_POLICY_TEMPERATURE,
    checkpoint_video_rollout_count=CHECKPOINT_VIDEO_ROLLOUT_COUNT,
    checkpoint_video_wandb_key_prefix=CHECKPOINT_VIDEO_WANDB_KEY_PREFIX,
)
FINAL_CHECKPOINT_PATH = training_result["final_checkpoint_path"]
ROLLOUT_CHECKPOINT_PATH = FINAL_CHECKPOINT_PATH
training_result


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/narf/.netrc.
wandb: Currently logged in as: fnattero (fnattero-universidad-de-san-andr-s) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/home/narf/miniconda3/envs/cool-antz/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training stage 1/1: 50x50_clusters_12_r00_sources_012
First update for this shape may compile; progress starts after it returns.


50x50_clusters_12_r00_sources_012: 500/10000 updates |▌         | 39:21<13:59:50 , loss=3.352, ret=-25.307 /home/narf/miniconda3/envs/cool-antz/lib/python3.10/subprocess.py:1796: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _posixsubprocess.fork_exec(
wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.
wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.
50x50_clusters_12_r00_sources_012: 1000/10000 updates |█         | 1:20:32<13:34:39 , loss=1.050, ret=-18.389wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.
wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.
50x50_clusters_12_r00_sources_012: 1500/10000 updates |█

KeyboardInterrupt: 

## Optional Local Render


In [ ]:
# rollout_result = workflows.render_jax_checkpoint_rollout(
#     run_dir=RUN_DIR,
#     checkpoint_path=BEST_CHECKPOINT_PATH,
#     media_dir=MEDIA_DIR,
#     rollout_filename="jax_mappo_exploration_to_forage_random_walls_rollout.mp4",
#     title="JAX MAPPO random wall curriculum rollout",
#     description="Sampled rollout from the random-wall lethal-cookie checkpoint.",
#     metadata={
#         "experiment_config": str(EXPERIMENT_CONFIG),
#         "source_checkpoint": str(SOURCE_CHECKPOINT),
#         "best_checkpoint": str(BEST_CHECKPOINT_PATH),
#         "num_ants": experiment_args.get("num_ants"),
#         "normal_food_sources": experiment_args.get("food_sources"),
#         "lethal_food_sources": experiment_args.get("lethal_food_sources"),
#         "lethal_food_min_distance": experiment_args.get("lethal_food_min_distance"),
#         "lethal_food_max_distance": experiment_args.get("lethal_food_max_distance"),
#         "random_wall_obstacles": experiment_args.get("random_wall_obstacles"),
#         "maze_layout_count": experiment_args.get("maze_layout_count"),
#         "random_wall_count_min": experiment_args.get("random_wall_count_min"),
#         "random_wall_count_max": experiment_args.get("random_wall_count_max"),
#         "random_wall_length_min": experiment_args.get("random_wall_length_min"),
#         "random_wall_length_max": experiment_args.get("random_wall_length_max"),
#         "random_wall_width": experiment_args.get("random_wall_width"),
#         "random_wall_l_turn_probability": experiment_args.get("random_wall_l_turn_probability"),
#         "random_wall_center_window_size": experiment_args.get("random_wall_center_window_size"),
#         "food_source_counts": [stage["food_sources"] for stage in CURRICULUM_STAGES],
#         "food_cluster_radii": [stage["food_cluster_radius"] for stage in CURRICULUM_STAGES],
#     },
#     tile_size=ROLLOUT_TILE_SIZE,
#     policy_temperature=ROLLOUT_POLICY_TEMPERATURE,
#     reuse_existing=False,
#     wandb_project=WANDB_PROJECT,
#     wandb_entity=WANDB_ENTITY,
#     wandb_group=WANDB_GROUP,
#     wandb_run_name=f"{WANDB_GROUP}_rollout",
#     wandb_mode="disabled",
#     wandb_video_key=None,
# )
# rollout_result
